# Module 14 — Notebook 1: Notebooks vs Scripts

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain when to use a notebook vs a Python script
- Extract a reusable function from notebook code into a `.py` file
- Apply the research workflow: explore → clean → extract → script

## Why This Matters for AI Research Engineering

In research, notebooks are for **exploration and communication**; scripts are for **pipelines, batch jobs, and reusable logic**. Knowing which to use — and how to transition between them — is a key habit that separates reproducible research from hard-to-repeat one-offs.

**JavaScript analogy:** A notebook is like a browser console session — great for trying things out interactively. A script is like a Node module — importable, testable, and reusable across projects.

At safety-focused labs, the ability to re-run an analysis from scratch (with a script) is often required before results are trusted.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_keys
print("Setup complete.")

## Notebooks vs Scripts — When to Use Each

| Use a **Notebook** when... | Use a **Script** when... |
|---|---|
| Exploring data for the first time | Running the same analysis on a schedule |
| Prototyping a new analysis | Processing a large batch of files |
| Creating figures and visualisations | Writing reusable helper functions |
| Writing a report or presenting results | Building a pipeline another tool calls |
| Iterating on a hypothesis interactively | The code needs to be imported by another module |

### The Research Workflow

A healthy workflow moves through four stages:

1. **Explore** — use a notebook to understand the data and try approaches
2. **Clean** — tidy up the working code: clear names, remove dead cells
3. **Extract** — pull reusable logic into a `.py` module
4. **Script** — write a standalone script (or pipeline) that calls those functions

Notebooks rarely disappear entirely — they are excellent for documenting *why* decisions were made.

## Worked Example — From Notebook to Module

Here is a function written inline in a notebook during exploration:

In [ ]:
# Inline notebook exploration
import json

def count_flagged(outputs):
    """Return the number of flagged outputs."""
    return sum(1 for r in outputs if r['flagged'])

# Try it on our data
with open('../../data/synthetic/model_outputs.json') as f:
    outputs = json.load(f)

print(f"Flagged: {count_flagged(outputs)} / {len(outputs)}")

Once `count_flagged` is working and tested, we can extract it into a `.py` file so other scripts and notebooks can import it. The cell below uses the `%%writefile` magic to write the file directly from the notebook — a handy bridge between exploration and extraction.

After extraction, any notebook can do `from analysis_utils import count_flagged` instead of copying the function.

## Exercise 1 — Classify Tasks

For each task below, decide whether it belongs in a **notebook** or a **script**.

Fill in each `'?'` with either `'notebook'` or `'script'`:

```python
task_types = {
    'explore_data': '?',      # trying new analysis ideas
    'run_daily_eval': '?',    # runs on a schedule, no human interaction
    'present_findings': '?',  # share results with the team
    'batch_process': '?'      # process 1000 files automatically
}
```

**Hints:**
- Anything running automatically (cron jobs, CI pipelines) should be a script
- Anything a human reads or interacts with is usually a notebook

In [ ]:
task_types = {
    'explore_data': '?',      # trying new analysis ideas
    'run_daily_eval': '?',    # runs on a schedule, no human interaction
    'present_findings': '?',  # share results with the team
    'batch_process': '?'      # process 1000 files automatically
}

In [ ]:
check_keys(task_types, ['explore_data', 'run_daily_eval', 'present_findings', 'batch_process'], "task_types has correct keys")
check_equal(task_types['run_daily_eval'], 'script', "run_daily_eval should be 'script'")
check_equal(task_types['batch_process'], 'script', "batch_process should be 'script'")
check_equal(task_types['explore_data'], 'notebook', "explore_data should be 'notebook'")
check_equal(task_types['present_findings'], 'notebook', "present_findings should be 'notebook'")

## Exercise 2 — Write a Reusable Loader

Write a function `load_and_count(path)` that:
1. Opens the JSON file at `path`
2. Loads it with `json.load()`
3. Returns `len(data)` — the number of records

This pattern (load + validate length) appears constantly in research pipelines.

In [ ]:
def load_and_count(path):
    # Your code here
    pass

In [ ]:
check_equal(load_and_count('../../data/synthetic/model_outputs.json'), 20, "load_and_count returns correct count")

## Exercise 3 — Extract to a Module

Now extract `load_and_count` into a standalone Python file called `analysis_utils.py`.

Use the `%%writefile` cell magic below. The magic writes the cell's content directly to a file — this is the standard notebook-to-script extraction technique.

After writing, we verify the file was created and contains the expected code.

In [ ]:
%%writefile analysis_utils.py
import json

def load_and_count(path):
    """Load a JSON file and return the number of records."""
    with open(path) as f:
        return len(json.load(f))

In [ ]:
source = Path('analysis_utils.py').read_text()

In [ ]:
check_equal(Path('analysis_utils.py').exists(), True, "analysis_utils.py was created")
check_contains(source, 'def load_and_count', "file contains the function definition")
check_contains(source, 'json.load', "file uses json.load")

## Wrap-Up

You now know when to reach for a notebook versus a script, and how to bridge the two using `%%writefile`. The key workflow to remember:

1. **Explore** in a notebook — fast iteration, visual feedback
2. **Extract** useful functions into `.py` modules using `%%writefile` or copy-paste
3. **Import** those modules back into notebooks (or other scripts) for reuse
4. **Script** anything that needs to run without human supervision

In the next notebook, we'll focus on writing the functions themselves more clearly — with good names, docstrings, and type hints.